# SHIVA-LLM · Objective 02 · Stage 3B
## Action Semantics and Temporal Transition Validation
**Runtime:** Python 3 · CPU (None) · 8–12 GB RAM recommended. No GPU needed.

**Required:** `objective02_stage3_reward_dataset.zip`. **Optional:** `skill_builder_data_corrected_collapsed.csv` for raw event adjacency checks.

**Safety:** This is an audit, **not** RL training. `logged_action` in Stage 1 is a shifted discretized question-difficulty category, not a verified instructional intervention. Stage 3 rewards are constructed proxies.

In [ ]:
import os, io, json, zipfile, pathlib, time
import numpy as np, pandas as pd
from google.colab import files
print('Upload objective02_stage3_reward_dataset.zip; optionally upload original ASSISTments CSV')
uploaded=files.upload()
zip_paths=[x for x in uploaded if x.lower().endswith('.zip')]
assert len(zip_paths)==1, f'Expected exactly one ZIP, got {zip_paths}'
zpath=zip_paths[0]
with zipfile.ZipFile(zpath) as z:
    names=z.namelist()
    print('ZIP contents:',names)
    candidates=[x for x in names if x.endswith('hyperkt_aligned_all_events.csv')]
    assert len(candidates)==1,'Expected hyperkt_aligned_all_events.csv'
    df=pd.read_csv(z.open(candidates[0]),low_memory=False)
    audit0=json.loads(z.read(next(x for x in names if x.endswith('stage3_audit.json'))))
print('Loaded:',len(df),'events; columns:',len(df.columns))
assert {'split','user_id','order_id','window_index','position','logged_action','target_correct','hyperkt_pred_correct'}.issubset(df.columns)

### 1 · Verify keys and learner partitions
The authoritative partitions are HyperKT's, not the incompatible Stage 1 partitions.

In [ ]:
assert not df.duplicated(['user_id','order_id']).any(), 'Duplicate original event IDs'
assert not df.duplicated(['split','window_index','position']).any(), 'Duplicate sequence positions'
assert df.groupby('user_id')['split'].nunique().max()==1, 'Learner split leakage'
assert df['hyperkt_pred_correct'].between(0,1).all(), 'Invalid prediction probability'
assert df['target_correct'].isin([0,1]).all(), 'Invalid binary outcome'
print(df.groupby('split').agg(events=('user_id','size'),learners=('user_id','nunique')).to_string())
print('Stage 1 split conflicts:',int(df['split_conflict'].fillna(False).sum()))

### 2 · Action semantics, coverage and support
**Known provenance:** Stage 1 computes per-question difficulty quantile bins, then shifts to the next recorded event. This may misalign the action with the current HyperKT target event. It is not an intervention log. We report distributions only.

In [ ]:
matched=df[df['has_logged_action'].fillna(False)].copy()
matched['logged_action']=pd.to_numeric(matched['logged_action'],errors='coerce')
assert matched['logged_action'].notna().all()
action_counts=pd.crosstab(matched['split'],matched['logged_action'])
print('Action category counts (shifted difficulty bins):\n',action_counts.to_string())
print('Proportions within each split:\n',action_counts.div(action_counts.sum(axis=1),axis=0).round(4).to_string())
print('Action bins per split:',matched.groupby('split')['logged_action'].nunique().to_dict())
print('Observed correctness by action (association only):')
print(matched.groupby(['split','logged_action'])['target_correct'].agg(['size','mean']).round(4).to_string())

### 3 · Temporal continuity audit
A difference in `order_id` is **not** an event-count gap; order IDs need not be consecutive. We check exact adjacency against the optional original CSV. Without it, continuity is **unverified**. Within HyperKT windows, sequence positions may be adjacent in the processed sequence but not necessarily in the original interaction log.

In [ ]:
df=df.sort_values(['user_id','order_id'],kind='mergesort').reset_index(drop=True)
next_user=df.groupby('user_id')['user_id'].shift(-1)
next_order=df.groupby('user_id')['order_id'].shift(-1)
next_window=df.groupby('user_id')['window_index'].shift(-1)
next_pos=df.groupby('user_id')['position'].shift(-1)
within_window=(df['user_id'].eq(next_user)&df['window_index'].eq(next_window)&next_pos.eq(df['position']+1))
print('Adjacent exported sequence positions:',int(within_window.sum()))
print('Non-adjacent or final exported rows:',int((~within_window).sum()))
print('Note: This does NOT prove original raw-event adjacency.')
raw_paths=[x for x in uploaded if x.lower().endswith('.csv')]
raw_adjacency_verified=False
if raw_paths:
    raw=None
    for enc in ['utf-8','cp1252','latin1']:
        try:
            raw=pd.read_csv(raw_paths[0],encoding=enc,usecols=['user_id','order_id'],low_memory=False)
            print('Raw CSV encoding:',enc);break
        except UnicodeDecodeError:continue
    assert raw is not None
    assert not raw.duplicated(['user_id','order_id']).any(),'Duplicate keys in raw data'
    raw=raw.sort_values(['user_id','order_id'],kind='mergesort').reset_index(drop=True)
    raw['raw_next_order_id']=raw.groupby('user_id')['order_id'].shift(-1)
    df=df.merge(raw[['user_id','order_id','raw_next_order_id']],on=['user_id','order_id'],how='left',validate='one_to_one',indicator='_raw_join')
    assert df['_raw_join'].eq('both').all(),'Some HyperKT events absent from original CSV'
    df['original_next_event_verified']=df['next_order_id'].eq(df['raw_next_order_id']) & df['next_order_id'].notna()
    raw_adjacency_verified=True
    print('Verified next raw event:',int(df['original_next_event_verified'].sum()),'/',len(df))
else:
    df['original_next_event_verified']=False
    print('Original CSV not uploaded: raw adjacency cannot be certified.')

### 4 · Check shifted-action mismatch and proxy limitations
We do not re-label shifted actions as current interventions. We also avoid using the current outcome as a pre-action state feature.

In [ ]:
# The Stage 1 logged action is defined from the NEXT event's question difficulty.
# Its mapping to this HyperKT target event was not established by the event crosswalk.
summary={
 'input_events':int(len(df)),
 'learners':int(df.user_id.nunique()),
 'matched_action_rows':int(len(matched)),
 'stage1_split_conflicts':int(df['split_conflict'].fillna(False).sum()),
 'hyperkt_partition_disjoint':bool(df.groupby('user_id')['split'].nunique().max()==1),
 'raw_adjacency_checked':bool(raw_adjacency_verified),
 'raw_adjacent_exported_transitions':int(df['original_next_event_verified'].sum()) if raw_adjacency_verified else None,
 'action_provenance':'Stage 1 next-event difficulty quantile bin, shifted to current row',
 'action_is_authenticated_intervention':False,
 'action_alignment_to_hyperkt_target_verified':False,
 'reward_is_observational_proxy':True,
 'offline_rl_policy_evaluation_ready':False,
 'next_step':'Recover actual item-level action/difficulty mapping or define an explicitly simulated instructional environment; audit temporal state construction and behavior support before RL claims'
}
print(json.dumps(summary,indent=2))

### 5 · Save audit bundle
Outputs are diagnostics, **not** a validated MDP dataset. The notebook intentionally refuses to certify offline RL readiness.

In [ ]:
out=pathlib.Path('/content/objective02_stage3b_outputs');out.mkdir(exist_ok=True)
with open(out/'stage3b_audit.json','w') as f:json.dump(summary,f,indent=2)
action_counts.to_csv(out/'action_counts.csv')
df.groupby('split').agg(events=('user_id','size'),learners=('user_id','nunique'),raw_adjacent_verified=('original_next_event_verified','sum')).to_csv(out/'split_transition_summary.csv')
df[['split','user_id','order_id','window_index','position','next_order_id','original_next_event_verified','has_logged_action','split_conflict']].to_csv(out/'event_transition_diagnostics.csv',index=False)
(out/'README.txt').write_text('Stage 3B audit only. Shifted difficulty bins are NOT authenticated interventions. Original raw-event adjacency is only verified when original CSV is provided. Offline RL readiness: false.\n')
zipout='/content/objective02_stage3b_action_transition_audit.zip'
with zipfile.ZipFile(zipout,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for f in out.iterdir():z.write(f,arcname=f.name)
print('Saved:',zipout,os.path.getsize(zipout)/1e6,'MB')
files.download(zipout)